# Task 4: Visual Search

### Imports and global setup

In [1]:
import copy
import json
import math
import random
from collections import Counter
from collections.abc import Callable, Iterable
from pathlib import Path
from typing import Any

import faiss
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn.functional as F
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from PIL import Image, ImageOps
from pytorch_metric_learning import losses
from torch import nn
from torch.utils.data import BatchSampler, DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import shutil
import zipfile

source_zip = Path("/content/drive/MyDrive/preprocessed_datasets.zip")
local_zip = Path("/content/preprocessed_datasets.zip")
extract_dir = Path("/content")

if not source_zip.exists():
    raise FileNotFoundError(f"File not found: {source_zip}")

shutil.copy2(source_zip, local_zip)
print(f"Copied to: {local_zip}")

with zipfile.ZipFile(local_zip, "r") as zip_file:
    zip_file.extractall(extract_dir)

print("Extracted to:", extract_dir)
local_zip.unlink()

Copied to: /content/preprocessed_datasets.zip
Extracted to: /content


In [6]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "preprocessed_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "preprocessed_datasets" / "train" / "styles_train.csv"
IMAGE_DIR = PROJECT_ROOT / "preprocessed_datasets" / "train" / "images_train"
SPLIT_DIR = PROJECT_ROOT / "splits" / "task4"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "task4"

CONFIG_DIR = ARTIFACT_DIR / "configs"
MODEL_DIR = ARTIFACT_DIR / "models"
OPTUNA_DB = ARTIFACT_DIR / "optuna.db"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
SEED = 42
RESNET_INPUT_SIZE = (128, 128)
INPUT_SIZE = RESNET_INPUT_SIZE

LABEL_COLUMN = "articleType_gender"
LABEL_ID_COLUMN = "articleType_gender_id"

MIN_CLASS_SIZE = 5
VAL_FRACTION = 0.10
CLASSES_PER_BATCH = 16
IMAGES_PER_CLASS = 4
CAE_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 256
NUM_WORKERS = 0

N_TRIALS = 20
TUNING_EPOCHS = 15
FINAL_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 10
MIN_DELTA = 1e-4

In [8]:
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [9]:
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"
PIN_MEMORY = AMP_ENABLED
print("Device:", DEVICE, "| Mixed precision:", AMP_ENABLED)

Device: cuda | Mixed precision: True


## 1. Data splitting

We first create the fixed outer train/holdout split, then derive an eligible inner development split without using the reserved holdout.

In [10]:
df = pd.read_csv(DATA_PATH)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37745 entries, 0 to 37744
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  37745 non-null  int64 
 1   gender              37745 non-null  object
 2   masterCategory      37745 non-null  object
 3   subCategory         37745 non-null  object
 4   articleType         37745 non-null  object
 5   baseColour          37745 non-null  object
 6   season              37745 non-null  object
 7   year                37745 non-null  int64 
 8   usage               37745 non-null  object
 9   productDisplayName  37745 non-null  object
dtypes: int64(2), object(8)
memory usage: 2.9+ MB


In [11]:
stratify_cols = [
    "articleType",
    "gender",
]

# One binary feature per category across all chosen columns
stratify_features = pd.get_dummies(
    df[stratify_cols].astype(str),
    prefix=stratify_cols,
)

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42,
)

outer_train_idx, test_idx = next(splitter.split(df, stratify_features))
outer_train_df = df.iloc[outer_train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train:", outer_train_df.shape)
print("Test: ", test_df.shape)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
outer_train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

Train: (33968, 10)
Test:  (3777, 10)


In [12]:
# Supervised metric-learning label: article type within its gender group.
# The mapping is fitted on the training partition only.
for split_df in (outer_train_df, test_df):
    split_df[LABEL_COLUMN] = (
        split_df["articleType"].str.strip() + "__" + split_df["gender"].str.strip()
    )

# JSON is portable and avoids serialising executable pickle/joblib objects.
classes = sorted(outer_train_df[LABEL_COLUMN].unique().tolist())
label_to_index = {label: index for index, label in enumerate(classes)}
outer_train_df[LABEL_ID_COLUMN] = outer_train_df[LABEL_COLUMN].map(label_to_index).astype("int64")
test_df[LABEL_ID_COLUMN] = test_df[LABEL_COLUMN].map(label_to_index).astype("Int64")

unseen_test_labels = sorted(set(test_df[LABEL_COLUMN]) - set(label_to_index))
if unseen_test_labels:
    print(
        f"Warning: {len(unseen_test_labels)} test label(s) are absent from training and have <NA> IDs."
    )
    print(unseen_test_labels)

with (CONFIG_DIR / "articleType_gender_label_encoder.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "label_column": LABEL_COLUMN,
            "label_id_column": LABEL_ID_COLUMN,
            "fit_split": "train",
            "separator": "__",
            "classes": classes,
            "label_to_index": label_to_index,
        },
        file,
        indent=2,
        sort_keys=True,
    )

class_counts = outer_train_df[LABEL_COLUMN].value_counts()
print(f"Saved {len(classes)} classes")
print(f"Classes with fewer than 2 training examples: {(class_counts < 2).sum()}")

['Innerwear Vests__Women', 'Shirts__Girls', 'Tracksuits__Women']
Saved 249 classes
Classes with fewer than 2 training examples: 30


## 2. Data preprocessing

Preprocessing is fitted from the outer training partition and reused consistently by every model.

### Image transforms and statistics

The original letterbox resize, RGB statistics, and transform definitions are retained.

In [13]:
class LetterboxResize:
    """Resize to fit within a canvas, padding instead of stretching or cropping."""

    def __init__(
        self, size: tuple[int, int], fill: tuple[int, int, int] = (255, 255, 255)
    ):
        self.size = tuple(size)
        self.fill = fill

    def __call__(self, image: Image.Image):
        return ImageOps.pad(
            image.convert("RGB"),
            self.size,
            method=Image.Resampling.BILINEAR,
            color=self.fill,
            centering=(0.5, 0.5),
        )


letterbox_to_resnet = LetterboxResize(RESNET_INPUT_SIZE)


def compute_rgb_mean_std(record_ids: Iterable[int]):
    """Calculate per-channel RGB statistics using training images only."""
    channel_sum = torch.zeros(3, dtype=torch.float64)
    channel_sum_sq = torch.zeros(3, dtype=torch.float64)
    pixel_count = 0

    for record_id in record_ids:
        path = IMAGE_DIR / f"{int(record_id)}.jpg"

        if not path.exists():
            raise FileNotFoundError(f"Missing image: {path}")

        with Image.open(path) as image:
            tensor = transforms.ToTensor()(letterbox_to_resnet(image)).to(torch.float64)

        channel_sum += tensor.sum(dim=(1, 2))
        channel_sum_sq += (tensor**2).sum(dim=(1, 2))
        pixel_count += tensor.shape[1] * tensor.shape[2]

    mean = channel_sum / pixel_count
    std = torch.sqrt(channel_sum_sq / pixel_count - mean**2)
    return mean.float().tolist(), std.float().tolist()


train_mean, train_std = compute_rgb_mean_std(outer_train_df["id"])
print("Training RGB mean:", np.round(train_mean, 4))
print("Training RGB std: ", np.round(train_std, 4))

Training RGB mean: [0.8862 0.8741 0.8695]
Training RGB std:  [0.2398 0.2513 0.2546]


In [14]:
image_preprocessing_config = {
    "input_color_mode": "RGB",
    "resize": {
        "method": "letterbox",
        "target_size": [128, 128],
        "interpolation": "bilinear",
        "padding_color_rgb": [255, 255, 255],
        "centering": [0.5, 0.5],
    },
    "tensor": {
        "layout": "CHW",
        "dtype": "float32",
        "value_range_before_normalization": [0.0, 1.0],
    },
    "normalization": {
        "mean_rgb": train_mean,
        "std_rgb": train_std,
    },
}

with (CONFIG_DIR / "image_preprocessing.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(image_preprocessing_config, file, indent=2)

### Transform assignment

- **CAE:** use `cae_transform` for both input and reconstruction target; start without augmentation.
- **Triplet Margin, Multi-Similarity, ArcFace:** use `metric_train_transform` during training and `metric_eval_transform` otherwise.
- **SupCon:** apply `metric_train_transform` twice independently to each training image for its two views; use `metric_eval_transform` for evaluation.

Random resized crops, strong rotations, and strong colour jitter are intentionally excluded because they can remove product details or distort colour information relevant to visual search.

In [15]:
# Triplet Margin, SupCon, Multi-Similarity, and ArcFace training.
# Mild geometric augmentation preserves the full product and its colour cues.
metric_train_transform = transforms.Compose([
    letterbox_to_resnet,
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        fill=(255, 255, 255),
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std),
])

# Use unchanged for validation, test, gallery, and query images.
metric_eval_transform = transforms.Compose([
    letterbox_to_resnet,
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std),
])

# CAE input and MSE reconstruction target are letterboxed to 128x128 in [0, 1].
# Pair this with a decoder ending in Sigmoid().
cae_transform = transforms.Compose([
    letterbox_to_resnet,
    transforms.ToTensor(),
])

In [16]:
metric_preprocessing_config = copy.deepcopy(image_preprocessing_config)
cae_preprocessing_config = copy.deepcopy(image_preprocessing_config)
cae_preprocessing_config.pop("normalization", None)

{'mean_rgb': [0.8862184286117554, 0.8741323947906494, 0.8695073127746582],
 'std_rgb': [0.23975732922554016, 0.251314252614975, 0.25460508465766907]}

### Datasets and batching

SupCon uses two transformed views; gallery and query loaders are deterministic.

In [17]:
class FashionImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, image_dir: str | Path, transform: Callable[[Image.Image], Any]):
        self.frame = frame.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        image_path = self.image_dir / f"{int(row['id'])}.jpg"
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            output = self.transform(image)

        label = row.get(LABEL_ID_COLUMN, pd.NA)
        label = -1 if pd.isna(label) else int(label)
        return {"image": output, "label": label, "id": int(row["id"])}

In [18]:
class PKBatchSampler(BatchSampler):
    def __init__(self, labels: Iterable[int], classes_per_batch: int, images_per_class: int, seed: int):
        self.labels = np.asarray(labels, dtype=np.int64)
        self.classes_per_batch = classes_per_batch
        self.images_per_class = images_per_class
        self.seed = seed
        self.epoch = 0
        self.class_indices = {
            label: np.flatnonzero(self.labels == label)
            for label in np.unique(self.labels)
        }
        self.batch_size = classes_per_batch * images_per_class
        self.batch_count = max(1, len(self.labels) // self.batch_size)

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __len__(self):
        return self.batch_count

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        available_classes = np.array(sorted(self.class_indices))

        for _ in range(self.batch_count):
            chosen_classes = rng.choice(
                available_classes,
                size=self.classes_per_batch,
                replace=False,
            )
            batch = []
            for label in chosen_classes:
                choices = self.class_indices[label]
                selected = rng.choice(choices, self.images_per_class, replace=False)
                batch.extend(selected.tolist())
            rng.shuffle(batch)
            yield batch


In [19]:
def seed_worker(worker_id: int):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def standard_loader(dataset: Dataset, batch_size: int, shuffle: bool = False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=NUM_WORKERS > 0,
    )

In [20]:
def make_two_view_transform(transform: Callable[[Image.Image], Any]):
    def apply(image: Image.Image):
        return transform(image), transform(image)

    return apply


def make_training_loader(model_name: str):
    transform = cae_transform if model_name == "cae" else metric_train_transform
    if model_name == "supcon":
        transform = make_two_view_transform(transform)
    dataset = FashionImageDataset(train_df, IMAGE_DIR, transform)
    if model_name == "cae":
        return standard_loader(dataset, CAE_BATCH_SIZE, shuffle=True)

    sampler = PKBatchSampler(
        train_df[LABEL_ID_COLUMN],
        CLASSES_PER_BATCH,
        IMAGES_PER_CLASS,
        SEED,
    )
    return DataLoader(
        dataset,
        batch_sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        worker_init_fn=seed_worker,
        persistent_workers=NUM_WORKERS > 0,
    )

def make_evaluation_loader(
    frame: pd.DataFrame,
    model_name: str,
):
    transform = cae_transform if model_name == "cae" else metric_eval_transform
    ordered_frame = frame.sort_values("id").reset_index(drop=True)
    dataset = FashionImageDataset(ordered_frame, IMAGE_DIR, transform)
    return standard_loader(dataset, EVAL_BATCH_SIZE, shuffle=False)

## 3. Model training

Each model is an independent experiment: tune its loss, inspect the best parameters, train a fresh model for longer with early stopping, validate on the inner split, and save its checkpoint and gallery arrays.

Training cells are intentionally separate and manually runnable. No global execution flags are used. Running setup cells does not start model training.

### Inner development split and validation retrieval setup

After preprocessing is defined, eligible outer-training rows are split into the inner training and validation partitions used for Optuna and early stopping. The reserved outer test partition remains unused.


In [ ]:
outer_class_counts = outer_train_df[LABEL_COLUMN].value_counts()
eligible_labels = set(
    outer_class_counts[outer_class_counts >= MIN_CLASS_SIZE].index
)
development_df = outer_train_df[
    outer_train_df[LABEL_COLUMN].isin(eligible_labels)
].copy()

inner_stratify_columns = ["articleType", "gender"]
inner_stratify_features = pd.get_dummies(
    development_df[inner_stratify_columns].astype(str),
    prefix=inner_stratify_columns,
)
inner_splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=VAL_FRACTION,
    random_state=SEED,
)
inner_train_idx, inner_val_idx = next(
    inner_splitter.split(development_df, inner_stratify_features)
)
train_df = development_df.iloc[inner_train_idx].copy()
val_df = development_df.iloc[inner_val_idx].copy()

In [ ]:
overlap = set(train_df["id"]) & set(val_df["id"])
if overlap:
    raise ValueError(f"Inner split contains {len(overlap)} overlapping IDs")
if val_df[LABEL_ID_COLUMN].isna().any():
    raise ValueError("Inner validation contains an unmapped label")

print(f"Model train/validation: {len(train_df):,}/{len(val_df):,}")
print("Excluded rare rows:", len(outer_train_df) - len(development_df))
print("Outer test rows kept aside:", len(test_df))

In [ ]:
gallery_parts = []
for _, group in train_df.groupby(LABEL_COLUMN, sort=True):
    sample_size = min(20, len(group))
    gallery_parts.append(group.sample(sample_size, random_state=SEED))

tuning_gallery_df = pd.concat(gallery_parts).sort_values("id").reset_index(drop=True)
print("Tuning gallery rows:", len(tuning_gallery_df))


In [ ]:
def make_model_loaders(model_name: str):
    return {
        "train": make_training_loader(model_name),
        "gallery": make_evaluation_loader(tuning_gallery_df, model_name),
        "query": make_evaluation_loader(val_df, model_name),
    }

### Shared ResNet-18 encoder

In [ ]:
class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        network = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(network.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, images: torch.Tensor):
        feature_map = self.features(images)
        embedding = self.pool(feature_map).flatten(1)
        return F.normalize(embedding, p=2, dim=1)

In [ ]:
def build_cae_decoder():
    channels = [512, 256, 128, 64, 32]
    blocks = []

    for input_channels, output_channels in zip(channels[:-1], channels[1:]):
        blocks.extend([
            nn.ConvTranspose2d(
                input_channels, output_channels, kernel_size=4, stride=2, padding=1
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True),
        ])

    blocks.extend([
        nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
        nn.Sigmoid(),
    ])
    return nn.Sequential(*blocks)


In [ ]:
class ConvolutionalAutoencoder(nn.Module):
    def __init__(self, encoder: ResNet18Encoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = build_cae_decoder()

    def forward(self, images: torch.Tensor):
        feature_map = self.encoder.features(images)
        embedding = self.encoder.pool(feature_map).flatten(1)
        reconstruction = self.decoder(feature_map)
        embedding = F.normalize(embedding, p=2, dim=1)
        return reconstruction, embedding

### Shared one-epoch training utility

In [ ]:
def compute_training_loss(model_name: str, model: nn.Module, loss_function: nn.Module, batch: dict[str, Any]):
    labels = batch["label"].to(DEVICE, non_blocking=True)

    if model_name == "supcon":
        first_view, second_view = batch["image"]
        images = torch.cat([first_view, second_view], dim=0).to(DEVICE)
        embeddings = model(images)
        return loss_function(embeddings, labels.repeat(2))

    images = batch["image"].to(DEVICE, non_blocking=True)
    if model_name == "cae":
        reconstruction, _ = model(images)
        return loss_function(reconstruction, images)
    if model_name == "arcface":
        return model.arcface_loss(model(images), labels)

    embeddings = model(images)
    return loss_function(embeddings, labels)


In [ ]:
def train_one_epoch(model_name: str, model: nn.Module, loss_function: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, scaler: torch.amp.GradScaler, epoch: int):
    model.train()
    if hasattr(loader.batch_sampler, "set_epoch"):
        loader.batch_sampler.set_epoch(epoch)

    running_loss = 0.0
    example_count = 0

    for batch in loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
            loss = compute_training_loss(model_name, model, loss_function, batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = len(batch["label"])
        running_loss += loss.detach().item() * batch_size
        example_count += batch_size

    return running_loss / max(1, example_count)


### Shared embedding and retrieval utilities

In [ ]:
def retrieval_embeddings(model: nn.Module, images: torch.Tensor):
    outputs = model(images)
    embeddings = outputs[1] if isinstance(outputs, tuple) else outputs
    return F.normalize(embeddings, dim=1)


def extract_embeddings(model: nn.Module, loader: DataLoader):
    model.eval()
    embeddings, identifiers, labels = [], [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                batch_embeddings = retrieval_embeddings(model, images)
            embeddings.append(batch_embeddings.cpu())
            identifiers.append(batch["id"].cpu())
            labels.append(batch["label"].cpu())

    return (
        torch.cat(embeddings).numpy().astype("float32"),
        torch.cat(identifiers).numpy().astype("int64"),
        torch.cat(labels).numpy().astype("int64"),
    )

In [ ]:
def build_faiss_index(gallery_embeddings: np.ndarray):
    embeddings = np.ascontiguousarray(gallery_embeddings, dtype="float32")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index


def search_cosine(query_embeddings: np.ndarray, gallery_embeddings: np.ndarray, maximum_k: int = 10):
    index = build_faiss_index(gallery_embeddings)
    search_k = min(maximum_k, len(gallery_embeddings))
    scores, indices = index.search(
        np.ascontiguousarray(query_embeddings, dtype="float32"),
        search_k,
    )
    return scores, indices, index

In [ ]:
def precision_k(
    ranked_indices: np.ndarray,
    query_labels: np.ndarray,
    gallery_labels: np.ndarray,
    k: int,
) -> float:
    effective_k = min(k, ranked_indices.shape[1])
    ranked_labels = gallery_labels[ranked_indices[:, :effective_k]]
    relevant = ranked_labels == query_labels[:, None]
    return float(relevant.mean(axis=1).mean())


In [ ]:
def recall_k(
    ranked_indices: np.ndarray,
    query_labels: np.ndarray,
    gallery_labels: np.ndarray,
    k: int,
) -> float:
    effective_k = min(k, ranked_indices.shape[1])
    ranked_labels = gallery_labels[ranked_indices[:, :effective_k]]
    relevant = ranked_labels == query_labels[:, None]
    gallery_counts = Counter(gallery_labels.tolist())
    relevant_counts = np.array([gallery_counts[int(label)] for label in query_labels])
    recalls = relevant.sum(axis=1) / np.maximum(relevant_counts, 1)
    return float(recalls.mean())


In [ ]:
def mean_average_precision_at_k(
    ranked_indices: np.ndarray,
    query_labels: np.ndarray,
    gallery_labels: np.ndarray,
    k: int,
) -> float:
    gallery_counts = Counter(gallery_labels.tolist())
    effective_k = min(k, ranked_indices.shape[1])
    average_precisions = []

    for row, query_label in zip(ranked_indices[:, :effective_k], query_labels):
        relevant = (gallery_labels[row] == query_label).astype(np.float32)
        cumulative_precision = np.cumsum(relevant) / np.arange(1, effective_k + 1)
        denominator = min(gallery_counts[int(query_label)], effective_k)
        average_precision = float((cumulative_precision * relevant).sum())
        average_precisions.append(average_precision / max(1, denominator))

    return float(np.mean(average_precisions))


In [ ]:
def retrieval_metrics(
    ranked_indices: np.ndarray,
    query_labels: np.ndarray,
    gallery_labels: np.ndarray,
    ks: tuple[int, ...] = (1, 5, 10),
) -> dict[str, float]:
    metrics = {}
    for k in ks:
        metrics[f"Precision@{k}"] = precision_k(
            ranked_indices, query_labels, gallery_labels, k
        )
        metrics[f"Recall@{k}"] = recall_k(
            ranked_indices, query_labels, gallery_labels, k
        )

    metrics["mAP@10"] = mean_average_precision_at_k(
        ranked_indices, query_labels, gallery_labels, 10
    )
    return metrics


In [ ]:
def evaluate_retrieval(
    model: nn.Module,
    gallery_loader: DataLoader,
    query_loader: DataLoader,
) -> dict[str, float]:
    gallery_embeddings, _, gallery_labels = extract_embeddings(
        model, gallery_loader
    )
    query_embeddings, _, query_labels = extract_embeddings(model, query_loader)
    _, ranked_indices, _ = search_cosine(
        query_embeddings, gallery_embeddings, maximum_k=10
    )

    return retrieval_metrics(ranked_indices, query_labels, gallery_labels)

### Shared early stopping and optimizer utilities

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int, minimum_delta: float):
        self.patience = patience
        self.minimum_delta = minimum_delta
        self.best_score = -math.inf
        self.bad_epochs = 0

    def update(self, score: float):
        improved = score > self.best_score + self.minimum_delta
        self.best_score = score if improved else self.best_score
        self.bad_epochs = 0 if improved else self.bad_epochs + 1
        return improved

    @property
    def should_stop(self):
        return self.bad_epochs >= self.patience


def cpu_state_dict(model: nn.Module):
    return {
        name: value.detach().cpu().clone()
        for name, value in model.state_dict().items()
    }


In [ ]:
def new_optimizer_and_scheduler(model: nn.Module, parameters: dict[str, Any]):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=parameters["learning_rate"],
        weight_decay=parameters["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )
    return optimizer, scheduler


### Shared Optuna setup

In [ ]:
def suggest_parameters(trial: optuna.Trial, model_name: str):
    parameters = {
        "learning_rate": trial.suggest_float(
            "learning_rate", 1e-5, 3e-3, log=True
        ),
        "weight_decay": trial.suggest_float(
            "weight_decay", 1e-6, 1e-3, log=True
        ),
    }

    if model_name == "triplet":
        parameters["margin"] = trial.suggest_float("margin", 0.1, 1.0, step=0.1)
    elif model_name == "supcon":
        parameters["temperature"] = trial.suggest_float(
            "temperature", 0.05, 0.20, step=0.025
        )
    elif model_name == "multi_similarity":
        parameters["alpha"] = trial.suggest_float("alpha", 1.0, 4.0)
        parameters["beta"] = trial.suggest_float("beta", 20.0, 80.0)
        parameters["base"] = trial.suggest_float("base", 0.3, 0.7)
    elif model_name == "arcface":
        parameters["margin"] = trial.suggest_float("margin", 0.1, 0.5)
        parameters["scale"] = trial.suggest_categorical(
            "scale", [16, 32, 48, 64]
        )
    return parameters


In [ ]:
def create_study(study_name: str):
    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    return optuna.create_study(
        study_name=study_name,
        storage=f"sqlite:///{OPTUNA_DB.resolve().as_posix()}",
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        load_if_exists=True,
    )


### Shared k-reciprocal re-ranking utility

In [ ]:
def gallery_reciprocal_sets(gallery_embeddings: np.ndarray, index: faiss.Index, k1: int):
    search_k = min(k1 + 1, len(gallery_embeddings))
    similarities, neighbours = index.search(gallery_embeddings, search_k)
    base_sets = []

    for gallery_index, row in enumerate(neighbours):
        forward = [int(item) for item in row if item != gallery_index][:k1]
        reciprocal = {
            candidate
            for candidate in forward
            if gallery_index in neighbours[candidate, 1:search_k]
        }
        base_sets.append(reciprocal)

    expanded_sets = []
    half_size = max(1, k1 // 2)
    for reciprocal in base_sets:
        expanded = set(reciprocal)
        for candidate in list(reciprocal):
            candidate_set = set(
                sorted(base_sets[candidate])[:half_size]
            )
            overlap = len(candidate_set & reciprocal)
            if candidate_set and overlap >= (2 * len(candidate_set) / 3):
                expanded.update(candidate_set)
        expanded_sets.append(expanded)

    thresholds = similarities[:, -1]
    return expanded_sets, thresholds


In [ ]:
def k_reciprocal_rerank(
    query_embeddings: np.ndarray,
    gallery_embeddings: np.ndarray,
    index: faiss.Index,
    k1: int = 20,
    k2: int = 6,
    blend: float = 0.3,
    maximum_k: int = 10,
    candidate_depth: int = 100,
):
    search_depth = min(candidate_depth, len(gallery_embeddings))
    initial_scores, candidates = index.search(query_embeddings, search_depth)
    gallery_sets, gallery_thresholds = gallery_reciprocal_sets(
        gallery_embeddings, index, k1
    )
    reranked_rows = []

    for scores, row in zip(initial_scores, candidates):
        forward_count = min(k1, len(row))
        forward = row[:forward_count]
        query_set = {
            int(candidate)
            for score, candidate in zip(scores[:forward_count], forward)
            if score >= gallery_thresholds[candidate]
        }
        for candidate in row[: min(k2, len(row))]:
            query_set.update(gallery_sets[int(candidate)])

        combined_distances = []
        for score, candidate in zip(scores, row):
            gallery_set = gallery_sets[int(candidate)]
            union = query_set | gallery_set
            intersection = query_set & gallery_set
            jaccard = 1.0 - len(intersection) / max(1, len(union))
            original_distance = 1.0 - float(score)
            distance = blend * original_distance + (1.0 - blend) * jaccard
            combined_distances.append(distance)

        ordering = np.argsort(combined_distances)[:maximum_k]
        reranked_rows.append(row[ordering])

    return np.asarray(reranked_rows, dtype=np.int64)


### 3.1 Convolutional Autoencoder

Run these cells in order. The study cell tunes hyperparameters, the final cell restarts training from fresh weights, and the artifact cell saves the checkpoint and gallery arrays. Record observations in the final markdown cell.

In [ ]:
def cae_objective(trial: optuna.Trial):
    parameters = suggest_parameters(trial, 'cae')
    set_seed(SEED)
    model = ConvolutionalAutoencoder(ResNet18Encoder())
    loss_function = nn.MSELoss()
    model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
    loaders = make_model_loaders('cae')
    optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_score = -math.inf
    for epoch in range(1, TUNING_EPOCHS + 1):
        train_one_epoch('cae', model, loss_function, loaders['train'], optimizer, scaler, epoch)
        score = evaluate_retrieval(model, loaders['gallery'], loaders['query'])['mAP@10']
        scheduler.step(score)
        best_score = max(best_score, score)
        trial.report(score, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_score

cae_study = create_study('task4_cae')
cae_study.optimize(cae_objective, n_trials=N_TRIALS)
cae_best_params = dict(cae_study.best_params)
print('cae', 'best parameters:', cae_best_params)

In [ ]:
cae_history = []
cae_best_epoch = 0
cae_best_score = -math.inf

parameters = cae_best_params
set_seed(SEED)
model = ConvolutionalAutoencoder(ResNet18Encoder())
loss_function = nn.MSELoss()
model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
loaders = make_model_loaders('cae')
optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
stopper = EarlyStopping(EARLY_STOPPING_PATIENCE, MIN_DELTA)
best_state = None
for epoch in tqdm(range(1, FINAL_EPOCHS + 1), desc='cae' + ' final'):
    train_loss = train_one_epoch('cae', model, loss_function, loaders['train'], optimizer, scaler, epoch)
    metrics = evaluate_retrieval(model, loaders['gallery'], loaders['query'])
    score = metrics['mAP@10']
    scheduler.step(score)
    if stopper.update(score):
        best_state = cpu_state_dict(model)
        cae_best_epoch = epoch
    cae_history.append({'epoch': epoch, 'train_loss': train_loss, **metrics})
    print('cae', epoch, 'loss:', round(train_loss, 4), 'mAP@10:', round(score, 4))
    if stopper.should_stop:
        break
model.load_state_dict(best_state)
cae_best_score = stopper.best_score
cae_model = model
print('cae', 'best epoch:', cae_best_epoch, 'best mAP@10:', cae_best_score)


In [ ]:
model_directory = MODEL_DIR / 'cae'
model_directory.mkdir(parents=True, exist_ok=True)
checkpoint = {
    'model_name': 'cae',
    'model_state_dict': cpu_state_dict(cae_model),
    'model_config': {
        'backbone': 'resnet18', 'pretrained': False,
        'input_size': list(INPUT_SIZE), 'embedding_dimension': 512,
        'class_count': len(classes),
    },
    'best_params': cae_best_params,
    'best_epoch': cae_best_epoch,
    'best_val_map_at_10': cae_best_score,
    'preprocessing_config': cae_preprocessing_config,
    'label_to_index': label_to_index,
    'gallery_scope': 'eligible_inner_training', 'random_seed': SEED,
}
torch.save(checkpoint, model_directory / 'best.pt')
gallery_loader = make_evaluation_loader(train_df, 'cae')
gallery_embeddings, gallery_ids, gallery_labels = extract_embeddings(cae_model, gallery_loader)
np.save(model_directory / 'gallery_embeddings.npy', gallery_embeddings)
np.save(model_directory / 'gallery_ids.npy', gallery_ids)
query_loader = make_evaluation_loader(val_df, 'cae')
query_embeddings, _, query_labels = extract_embeddings(cae_model, query_loader)
_, base_indices, index = search_cosine(query_embeddings, gallery_embeddings)
reranked_indices = k_reciprocal_rerank(query_embeddings, gallery_embeddings, index)
print('cae', 'base:', retrieval_metrics(base_indices, query_labels, gallery_labels))
print('cae', 'reranked:', retrieval_metrics(reranked_indices, query_labels, gallery_labels))
print('Saved:', model_directory / 'best.pt')

Observation note: record best validation mAP@10, convergence behavior, and artifact paths here.

### 3.2 Triplet Margin Loss

Run these cells in order. The study cell tunes hyperparameters, the final cell restarts training from fresh weights, and the artifact cell saves the checkpoint and gallery arrays. Record observations in the final markdown cell.

In [ ]:
def triplet_objective(trial: optuna.Trial):
    parameters = suggest_parameters(trial, 'triplet')
    set_seed(SEED)
    model = ResNet18Encoder()
    loss_function = losses.TripletMarginLoss(margin=parameters.get('margin', 0.3))
    model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
    loaders = make_model_loaders('triplet')
    optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_score = -math.inf
    for epoch in range(1, TUNING_EPOCHS + 1):
        train_one_epoch('triplet', model, loss_function, loaders['train'], optimizer, scaler, epoch)
        score = evaluate_retrieval(model, loaders['gallery'], loaders['query'])['mAP@10']
        scheduler.step(score)
        best_score = max(best_score, score)
        trial.report(score, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_score

triplet_study = create_study('task4_triplet')
triplet_study.optimize(triplet_objective, n_trials=N_TRIALS)
triplet_best_params = dict(triplet_study.best_params)
print('triplet', 'best parameters:', triplet_best_params)

In [ ]:
triplet_history = []
triplet_best_epoch = 0
triplet_best_score = -math.inf

parameters = triplet_best_params
set_seed(SEED)
model = ResNet18Encoder()
loss_function = losses.TripletMarginLoss(margin=parameters.get('margin', 0.3))
model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
loaders = make_model_loaders('triplet')
optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
stopper = EarlyStopping(EARLY_STOPPING_PATIENCE, MIN_DELTA)
best_state = None
for epoch in tqdm(range(1, FINAL_EPOCHS + 1), desc='triplet' + ' final'):
    train_loss = train_one_epoch('triplet', model, loss_function, loaders['train'], optimizer, scaler, epoch)
    metrics = evaluate_retrieval(model, loaders['gallery'], loaders['query'])
    score = metrics['mAP@10']
    scheduler.step(score)
    if stopper.update(score):
        best_state = cpu_state_dict(model)
        triplet_best_epoch = epoch
    triplet_history.append({'epoch': epoch, 'train_loss': train_loss, **metrics})
    print('triplet', epoch, 'loss:', round(train_loss, 4), 'mAP@10:', round(score, 4))
    if stopper.should_stop:
        break
model.load_state_dict(best_state)
triplet_best_score = stopper.best_score
triplet_model = model
print('triplet', 'best epoch:', triplet_best_epoch, 'best mAP@10:', triplet_best_score)


In [ ]:
model_directory = MODEL_DIR / 'triplet'
model_directory.mkdir(parents=True, exist_ok=True)
checkpoint = {
    'model_name': 'triplet',
    'model_state_dict': cpu_state_dict(triplet_model),
    'model_config': {
        'backbone': 'resnet18', 'pretrained': False,
        'input_size': list(INPUT_SIZE), 'embedding_dimension': 512,
        'class_count': len(classes),
    },
    'best_params': triplet_best_params,
    'best_epoch': triplet_best_epoch,
    'best_val_map_at_10': triplet_best_score,
    'preprocessing_config': metric_preprocessing_config,
    'label_to_index': label_to_index,
    'gallery_scope': 'eligible_inner_training', 'random_seed': SEED,
}
torch.save(checkpoint, model_directory / 'best.pt')
gallery_loader = make_evaluation_loader(train_df, 'triplet')
gallery_embeddings, gallery_ids, gallery_labels = extract_embeddings(triplet_model, gallery_loader)
np.save(model_directory / 'gallery_embeddings.npy', gallery_embeddings)
np.save(model_directory / 'gallery_ids.npy', gallery_ids)
query_loader = make_evaluation_loader(val_df, 'triplet')
query_embeddings, _, query_labels = extract_embeddings(triplet_model, query_loader)
_, base_indices, index = search_cosine(query_embeddings, gallery_embeddings)
reranked_indices = k_reciprocal_rerank(query_embeddings, gallery_embeddings, index)
print('triplet', 'base:', retrieval_metrics(base_indices, query_labels, gallery_labels))
print('triplet', 'reranked:', retrieval_metrics(reranked_indices, query_labels, gallery_labels))
print('Saved:', model_directory / 'best.pt')

Observation note: record best validation mAP@10, convergence behavior, and artifact paths here.

### 3.3 Supervised Contrastive Loss

Run these cells in order. The study cell tunes hyperparameters, the final cell restarts training from fresh weights, and the artifact cell saves the checkpoint and gallery arrays. Record observations in the final markdown cell.

In [ ]:
def supcon_objective(trial: optuna.Trial):
    parameters = suggest_parameters(trial, 'supcon')
    set_seed(SEED)
    model = ResNet18Encoder()
    loss_function = losses.SupConLoss(temperature=parameters.get('temperature', 0.1))
    model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
    loaders = make_model_loaders('supcon')
    optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_score = -math.inf
    for epoch in range(1, TUNING_EPOCHS + 1):
        train_one_epoch('supcon', model, loss_function, loaders['train'], optimizer, scaler, epoch)
        score = evaluate_retrieval(model, loaders['gallery'], loaders['query'])['mAP@10']
        scheduler.step(score)
        best_score = max(best_score, score)
        trial.report(score, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_score

supcon_study = create_study('task4_supcon')
supcon_study.optimize(supcon_objective, n_trials=N_TRIALS)
supcon_best_params = dict(supcon_study.best_params)
print('supcon', 'best parameters:', supcon_best_params)

In [ ]:
supcon_history = []
supcon_best_epoch = 0
supcon_best_score = -math.inf

parameters = supcon_best_params
set_seed(SEED)
model = ResNet18Encoder()
loss_function = losses.SupConLoss(temperature=parameters.get('temperature', 0.1))
model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
loaders = make_model_loaders('supcon')
optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
stopper = EarlyStopping(EARLY_STOPPING_PATIENCE, MIN_DELTA)
best_state = None
for epoch in tqdm(range(1, FINAL_EPOCHS + 1), desc='supcon' + ' final'):
    train_loss = train_one_epoch('supcon', model, loss_function, loaders['train'], optimizer, scaler, epoch)
    metrics = evaluate_retrieval(model, loaders['gallery'], loaders['query'])
    score = metrics['mAP@10']
    scheduler.step(score)
    if stopper.update(score):
        best_state = cpu_state_dict(model)
        supcon_best_epoch = epoch
    supcon_history.append({'epoch': epoch, 'train_loss': train_loss, **metrics})
    print('supcon', epoch, 'loss:', round(train_loss, 4), 'mAP@10:', round(score, 4))
    if stopper.should_stop:
        break
model.load_state_dict(best_state)
supcon_best_score = stopper.best_score
supcon_model = model
print('supcon', 'best epoch:', supcon_best_epoch, 'best mAP@10:', supcon_best_score)


In [ ]:
model_directory = MODEL_DIR / 'supcon'
model_directory.mkdir(parents=True, exist_ok=True)
checkpoint = {
    'model_name': 'supcon',
    'model_state_dict': cpu_state_dict(supcon_model),
    'model_config': {
        'backbone': 'resnet18', 'pretrained': False,
        'input_size': list(INPUT_SIZE), 'embedding_dimension': 512,
        'class_count': len(classes),
    },
    'best_params': supcon_best_params,
    'best_epoch': supcon_best_epoch,
    'best_val_map_at_10': supcon_best_score,
    'preprocessing_config': metric_preprocessing_config,
    'label_to_index': label_to_index,
    'gallery_scope': 'eligible_inner_training', 'random_seed': SEED,
}
torch.save(checkpoint, model_directory / 'best.pt')
gallery_loader = make_evaluation_loader(train_df, 'supcon')
gallery_embeddings, gallery_ids, gallery_labels = extract_embeddings(supcon_model, gallery_loader)
np.save(model_directory / 'gallery_embeddings.npy', gallery_embeddings)
np.save(model_directory / 'gallery_ids.npy', gallery_ids)
query_loader = make_evaluation_loader(val_df, 'supcon')
query_embeddings, _, query_labels = extract_embeddings(supcon_model, query_loader)
_, base_indices, index = search_cosine(query_embeddings, gallery_embeddings)
reranked_indices = k_reciprocal_rerank(query_embeddings, gallery_embeddings, index)
print('supcon', 'base:', retrieval_metrics(base_indices, query_labels, gallery_labels))
print('supcon', 'reranked:', retrieval_metrics(reranked_indices, query_labels, gallery_labels))
print('Saved:', model_directory / 'best.pt')

Observation note: record best validation mAP@10, convergence behavior, and artifact paths here.

### 3.4 Multi-Similarity Loss

Run these cells in order. The study cell tunes hyperparameters, the final cell restarts training from fresh weights, and the artifact cell saves the checkpoint and gallery arrays. Record observations in the final markdown cell.

In [ ]:
def multi_similarity_objective(trial: optuna.Trial):
    parameters = suggest_parameters(trial, 'multi_similarity')
    set_seed(SEED)
    model = ResNet18Encoder()
    loss_function = losses.MultiSimilarityLoss(alpha=parameters.get('alpha', 2.0), beta=parameters.get('beta', 50.0), base=parameters.get('base', 0.5))
    model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
    loaders = make_model_loaders('multi_similarity')
    optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_score = -math.inf
    for epoch in range(1, TUNING_EPOCHS + 1):
        train_one_epoch('multi_similarity', model, loss_function, loaders['train'], optimizer, scaler, epoch)
        score = evaluate_retrieval(model, loaders['gallery'], loaders['query'])['mAP@10']
        scheduler.step(score)
        best_score = max(best_score, score)
        trial.report(score, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_score

multi_similarity_study = create_study('task4_multi_similarity')
multi_similarity_study.optimize(multi_similarity_objective, n_trials=N_TRIALS)
multi_similarity_best_params = dict(multi_similarity_study.best_params)
print('multi_similarity', 'best parameters:', multi_similarity_best_params)

In [ ]:
multi_similarity_history = []
multi_similarity_best_epoch = 0
multi_similarity_best_score = -math.inf

parameters = multi_similarity_best_params
set_seed(SEED)
model = ResNet18Encoder()
loss_function = losses.MultiSimilarityLoss(alpha=parameters.get('alpha', 2.0), beta=parameters.get('beta', 50.0), base=parameters.get('base', 0.5))
model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
loaders = make_model_loaders('multi_similarity')
optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
stopper = EarlyStopping(EARLY_STOPPING_PATIENCE, MIN_DELTA)
best_state = None
for epoch in tqdm(range(1, FINAL_EPOCHS + 1), desc='multi_similarity' + ' final'):
    train_loss = train_one_epoch('multi_similarity', model, loss_function, loaders['train'], optimizer, scaler, epoch)
    metrics = evaluate_retrieval(model, loaders['gallery'], loaders['query'])
    score = metrics['mAP@10']
    scheduler.step(score)
    if stopper.update(score):
        best_state = cpu_state_dict(model)
        multi_similarity_best_epoch = epoch
    multi_similarity_history.append({'epoch': epoch, 'train_loss': train_loss, **metrics})
    print('multi_similarity', epoch, 'loss:', round(train_loss, 4), 'mAP@10:', round(score, 4))
    if stopper.should_stop:
        break
model.load_state_dict(best_state)
multi_similarity_best_score = stopper.best_score
multi_similarity_model = model
print('multi_similarity', 'best epoch:', multi_similarity_best_epoch, 'best mAP@10:', multi_similarity_best_score)


In [ ]:
model_directory = MODEL_DIR / 'multi_similarity'
model_directory.mkdir(parents=True, exist_ok=True)
checkpoint = {
    'model_name': 'multi_similarity',
    'model_state_dict': cpu_state_dict(multi_similarity_model),
    'model_config': {
        'backbone': 'resnet18', 'pretrained': False,
        'input_size': list(INPUT_SIZE), 'embedding_dimension': 512,
        'class_count': len(classes),
    },
    'best_params': multi_similarity_best_params,
    'best_epoch': multi_similarity_best_epoch,
    'best_val_map_at_10': multi_similarity_best_score,
    'preprocessing_config': metric_preprocessing_config,
    'label_to_index': label_to_index,
    'gallery_scope': 'eligible_inner_training', 'random_seed': SEED,
}
torch.save(checkpoint, model_directory / 'best.pt')
gallery_loader = make_evaluation_loader(train_df, 'multi_similarity')
gallery_embeddings, gallery_ids, gallery_labels = extract_embeddings(multi_similarity_model, gallery_loader)
np.save(model_directory / 'gallery_embeddings.npy', gallery_embeddings)
np.save(model_directory / 'gallery_ids.npy', gallery_ids)
query_loader = make_evaluation_loader(val_df, 'multi_similarity')
query_embeddings, _, query_labels = extract_embeddings(multi_similarity_model, query_loader)
_, base_indices, index = search_cosine(query_embeddings, gallery_embeddings)
reranked_indices = k_reciprocal_rerank(query_embeddings, gallery_embeddings, index)
print('multi_similarity', 'base:', retrieval_metrics(base_indices, query_labels, gallery_labels))
print('multi_similarity', 'reranked:', retrieval_metrics(reranked_indices, query_labels, gallery_labels))
print('Saved:', model_directory / 'best.pt')

Observation note: record best validation mAP@10, convergence behavior, and artifact paths here.

### 3.5 ArcFace

Run these cells in order. The study cell tunes hyperparameters, the final cell restarts training from fresh weights, and the artifact cell saves the checkpoint and gallery arrays. Record observations in the final markdown cell.

In [ ]:
def arcface_objective(trial: optuna.Trial):
    parameters = suggest_parameters(trial, 'arcface')
    set_seed(SEED)
    model = ResNet18Encoder()
    model.arcface_loss = losses.ArcFaceLoss(
        num_classes=len(classes),
        embedding_size=512,
        margin=math.degrees(parameters.get('margin', 0.3)),
        scale=parameters.get('scale', 32),
    )
    loss_function = model.arcface_loss
    model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
    loaders = make_model_loaders('arcface')
    optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_score = -math.inf
    for epoch in range(1, TUNING_EPOCHS + 1):
        train_one_epoch('arcface', model, loss_function, loaders['train'], optimizer, scaler, epoch)
        score = evaluate_retrieval(model, loaders['gallery'], loaders['query'])['mAP@10']
        scheduler.step(score)
        best_score = max(best_score, score)
        trial.report(score, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_score

arcface_study = create_study('task4_arcface')
arcface_study.optimize(arcface_objective, n_trials=N_TRIALS)
arcface_best_params = dict(arcface_study.best_params)
print('arcface', 'best parameters:', arcface_best_params)

In [ ]:
arcface_history = []
arcface_best_epoch = 0
arcface_best_score = -math.inf

parameters = arcface_best_params
set_seed(SEED)
model = ResNet18Encoder()
model.arcface_loss = losses.ArcFaceLoss(
    num_classes=len(classes),
    embedding_size=512,
    margin=math.degrees(parameters.get('margin', 0.3)),
    scale=parameters.get('scale', 32),
)
loss_function = model.arcface_loss
model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
loaders = make_model_loaders('arcface')
optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
stopper = EarlyStopping(EARLY_STOPPING_PATIENCE, MIN_DELTA)
best_state = None
for epoch in tqdm(range(1, FINAL_EPOCHS + 1), desc='arcface' + ' final'):
    train_loss = train_one_epoch('arcface', model, loss_function, loaders['train'], optimizer, scaler, epoch)
    metrics = evaluate_retrieval(model, loaders['gallery'], loaders['query'])
    score = metrics['mAP@10']
    scheduler.step(score)
    if stopper.update(score):
        best_state = cpu_state_dict(model)
        arcface_best_epoch = epoch
    arcface_history.append({'epoch': epoch, 'train_loss': train_loss, **metrics})
    print('arcface', epoch, 'loss:', round(train_loss, 4), 'mAP@10:', round(score, 4))
    if stopper.should_stop:
        break
model.load_state_dict(best_state)
arcface_best_score = stopper.best_score
arcface_model = model
print('arcface', 'best epoch:', arcface_best_epoch, 'best mAP@10:', arcface_best_score)


In [ ]:
model_directory = MODEL_DIR / 'arcface'
model_directory.mkdir(parents=True, exist_ok=True)
checkpoint = {
    'model_name': 'arcface',
    'model_state_dict': cpu_state_dict(arcface_model),
    'model_config': {
        'backbone': 'resnet18', 'pretrained': False,
        'input_size': list(INPUT_SIZE), 'embedding_dimension': 512,
        'class_count': len(classes),
    },
    'best_params': arcface_best_params,
    'best_epoch': arcface_best_epoch,
    'best_val_map_at_10': arcface_best_score,
    'preprocessing_config': metric_preprocessing_config,
    'label_to_index': label_to_index,
    'gallery_scope': 'eligible_inner_training', 'random_seed': SEED,
}
torch.save(checkpoint, model_directory / 'best.pt')
gallery_loader = make_evaluation_loader(train_df, 'arcface')
gallery_embeddings, gallery_ids, gallery_labels = extract_embeddings(arcface_model, gallery_loader)
np.save(model_directory / 'gallery_embeddings.npy', gallery_embeddings)
np.save(model_directory / 'gallery_ids.npy', gallery_ids)
query_loader = make_evaluation_loader(val_df, 'arcface')
query_embeddings, _, query_labels = extract_embeddings(arcface_model, query_loader)
_, base_indices, index = search_cosine(query_embeddings, gallery_embeddings)
reranked_indices = k_reciprocal_rerank(query_embeddings, gallery_embeddings, index)
print('arcface', 'base:', retrieval_metrics(base_indices, query_labels, gallery_labels))
print('arcface', 'reranked:', retrieval_metrics(reranked_indices, query_labels, gallery_labels))
print('Saved:', model_directory / 'best.pt')

Observation note: record best validation mAP@10, convergence behavior, and artifact paths here.

## 4. Evaluation on test set (deferred)

No test evaluation is performed now. After model selection is authorized, load the reserved outer holdout, encode it with the selected checkpoint, and calculate base and reranked retrieval metrics.

## 5. Compare and finalize model (deferred)

No comparison or final model selection is performed now. A later stage should compare mAP@10, Precision@K, Recall@K, training cost, and artifact completeness.